In [ ]:
cd ../..

In [ ]:
import yaml, sys
import pandas as pd
import numpy as np
import plotly.express as px
from src.feature_importance import FeatureImportance
from src.reduce_dimensions import ReduceDimensions
from loguru import logger
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

sns.set_context("poster")
sns.set_style("ticks")

np.random.seed(0)

logger.remove()
logger.add(sys.stderr, level="WARNING")

# Interactions

In [ ]:
all_importances = []
all_spearman = []
all_weights = {}
for param, method in [
    ("cholesky", "MLEM"),
    ("triu", "FR-RSA"),
]:
    cfg = f"""
    dataset:
        path: datasets/simulated.csv
    trainer:
        model_builder:
            param: {param}
        representations:
            level: simulated
    """
    cfg = yaml.safe_load(cfg)
    fi = FeatureImportance(**cfg)
    i, s, w = fi.compute()
    i["Method"] = method
    s["Method"] = method
    all_importances.append(i)
    all_spearman.append(s)
    model, _ = fi.trainer.init(fi.trainer.train()[0])
    all_weights[method] = model.get_formatted_W()
all_importances = pd.concat(all_importances)
all_spearman = pd.concat(all_spearman)

In [ ]:
sns.heatmap(
    all_weights["FR-RSA dummies"],
    cmap="coolwarm",
    center=0,
)

In [ ]:
g = sns.FacetGrid(all_importances, col="Method", hue="Feature", height=6, aspect=1.5)
g.map_dataframe(
    sns.barplot,
    y="Feature",
    x="mean",
    hue="Feature",
    hue_order=all_importances["Feature"].unique(),
    palette="tab10",
    orient="h",
)
g.map_dataframe(
    plt.errorbar,
    y="Feature",
    x="mean",
    xerr="std",
    linewidth=3,
    capsize=5,
    capthick=3,
    color="k",
)
g.set_xlabels("Feature Importance", labelpad=20)
g.set_ylabels("Feature", labelpad=30)

for ax, method in zip(g.axes.flat, all_importances["Method"].unique()):
    spearman_mean = all_spearman[all_spearman["Method"] == method]["mean"].values[0]
    spearman_std = all_spearman[all_spearman["Method"] == method]["std"].values[0]
    ax.set_title(f"{method}\nSpearman: {spearman_mean:.2f} ± {spearman_std:.1g}", pad=30)
    ax.tick_params(axis="y", length=0)

sns.despine(trim=True, offset=10, left=True)
plt.tight_layout()
plt.savefig(
    "experiments/simulations/feature_importance.pdf", bbox_inches="tight", pad_inches=0
)
plt.show()

In [ ]:
rd = ReduceDimensions(
    dataset=cfg["dataset"],
    representations=cfg["trainer"]["representations"],
    method="none",
)
df = rd.transform()
df["Interaction:\nFeat. 1 == Feat. 2"] = df["Feat. 1"] == df["Feat. 2"]
tmp = df.melt(
    id_vars=[0, 1],
    value_vars=["Feat. 1", "Feat. 2", "Feat. 3", "Interaction:\nFeat. 1 == Feat. 2"],
    var_name="Feature",
    value_name="Value",
)

In [ ]:
g = sns.FacetGrid(tmp, col="Feature", col_wrap=2, height=4, aspect=1.2)
g.map_dataframe(
    sns.scatterplot,
    x=0,
    y=1,
    hue="Value",
    alpha=0.7,
)
# g.set_axis_labels("Dimension 0", "Dimension 1", labelpad=15)
g.set_titles(col_template="Feature: {col_name}", pad=15)
for ax in g.axes.flat:
    ax.spines["top"].set_visible(True)
    ax.spines["right"].set_visible(True)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel("")
    ax.set_ylabel("")
plt.tight_layout()
plt.savefig("experiments/simulations/dataset.pdf", bbox_inches="tight", pad_inches=0)
plt.show()

# Ground-truth experiment

In [ ]:
gt = pd.DataFrame(
    {
        "Feature": [
            "Feat. 1",
            "Feat. 2",
            "(Feat. 1 x Feat. 2)",
        ],
        "GTWeight": [
            0.6324,
            0.6324,
            -0.6324,
        ],
    }
)

In [ ]:
methods = [
    ("FR-RSA", "triu"),
    ("MLEM", "cholesky"),
]
noise_levels = [0, 0.1, 0.2, 0.3]
budgets = [10, 30, 50, 100, 200, 300, 500, 750, 1000]
results = []
with tqdm(total=len(methods) * len(noise_levels) * len(budgets)) as pbar:
    for method, param in methods:
        for noise_level in noise_levels:
            for budget in budgets:
                cfg = f"""
                dataset:
                    path: datasets/simulated.csv
                trainer:
                    max_epochs: {budget}
                    dataloader_builder:
                        cv: 3
                    model_builder:
                        param: {param}
                    representations:
                        level: simulated
                        noise_level: {noise_level}
                """
                cfg = yaml.safe_load(cfg)
                fi = FeatureImportance(**cfg)
                i, s, w = fi.compute()
                s = s[s.split == "test"]
                w = w.merge(gt, how="left").fillna(0)
                weight_diff = (
                    w.groupby("cv")
                    .apply(
                        lambda x: np.linalg.norm(x.Weight - x.GTWeight),
                        include_groups=False,
                    )
                    .reset_index(drop=True)
                    .to_frame(name="mean")
                )
                for df, variable in [
                    (weight_diff, "Weight dist. to gt"),
                    (s, "Encoding Spearman"),
                ]:
                    df["Noise level"] = noise_level
                    df["variable"] = variable
                    df["Method"] = method
                    df["Budget"] = budget
                    results.append(df)
                pbar.update(1)
results = pd.concat(results)

In [ ]:
g = sns.relplot(
    results,
    x="Budget",
    y="mean",
    hue="Method",
    row="Noise level",
    col="variable",
    kind="line",
    aspect=1.75,
    height=4,
    facet_kws={"sharey": False},
    marker="o",
    markersize=5,
)
g.set(xscale="log")
g.set_titles("{col_name} | {row_name}")
g.set_axis_labels("Max epochs", "")